In [32]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

BASE_DIR = Path(
    r"D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI"
)

RAW_DIR = BASE_DIR / "datasets" / "raw"
OUT_DIR = BASE_DIR / "datasets" / "processed"
REPORT_DIR = BASE_DIR / "datasets" / "reports"

OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder exists:", RAW_DIR.exists())
print("Output folder:", OUT_DIR)
print("Reports folder:", REPORT_DIR)

Raw folder exists: True
Output folder: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\processed
Reports folder: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\reports


In [33]:
RAW_FILE = RAW_DIR / "perishable_inventory.csv"

if not RAW_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found: {RAW_FILE}"
    )

df = pd.read_csv(
    RAW_FILE,
    dtype={"product_id": str}
)

print("Dataset loaded successfully")
print("Rows and columns:", df.shape)

display(df.head())

Dataset loaded successfully
Rows and columns: (1800, 12)


,product_id,date,category,opening_stock,units_received,units_sold,unit_price,discount_pct,days_to_expiry,temperature_c,waste_units,stockout_flag
0,1001,2026-04-01,Dairy,27,0,6,32.0,0.0,1,4.1,0,0
1,1001,2026-04-02,Dairy,30,12,5,32.0,0.0,5,3.7,0,0
2,1001,2026-04-03,Dairy,38,0,14,32.0,0.0,2,4.6,0,0
3,1001,2026-04-04,Dairy,49,0,11,32.0,0.0,0,5.7,0,0
4,1001,2026-04-05,Dairy,19,13,12,32.0,0.0,0,3.0,0,0


In [34]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values in each column:")
display(df.isnull().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())

Column names:
['product_id', 'date', 'category', 'opening_stock', 'units_received', 'units_sold', 'unit_price', 'discount_pct', 'days_to_expiry', 'temperature_c', 'waste_units', 'stockout_flag']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      1800 non-null   str    
 1   date            1800 non-null   str    
 2   category        1800 non-null   str    
 3   opening_stock   1800 non-null   int64  
 4   units_received  1800 non-null   int64  
 5   units_sold      1800 non-null   int64  
 6   unit_price      1800 non-null   float64
 7   discount_pct    1800 non-null   float64
 8   days_to_expiry  1800 non-null   int64  
 9   temperature_c   1800 non-null   float64
 10  waste_units     1800 non-null   int64  
 11  stockout_flag   1800 non-null   int64  
dtypes: float64(3), int64(6), str(3)
memory usage: 204.2 

,missing_values
product_id,0
date,0
category,0
opening_stock,0
units_received,0
units_sold,0
unit_price,0
discount_pct,0
days_to_expiry,0
temperature_c,0



Duplicate rows: 0


In [35]:
df["product_id"] = df["product_id"].astype("string").str.strip()
df["category"] = df["category"].astype("string").str.strip()

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

print("Date conversion completed")
print("Invalid dates:", df["date"].isna().sum())

display(df[["product_id", "category", "date"]].head())

Date conversion completed
Invalid dates: 0


,product_id,category,date
0,1001,Dairy,2026-04-01
1,1001,Dairy,2026-04-02
2,1001,Dairy,2026-04-03
3,1001,Dairy,2026-04-04
4,1001,Dairy,2026-04-05


In [36]:
numeric_columns = [
    "opening_stock",
    "units_received",
    "units_sold",
    "unit_price",
    "discount_pct",
    "days_to_expiry",
    "temperature_c",
    "waste_units",
    "stockout_flag"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

print("Numeric conversion completed")

display(df[numeric_columns].head())

Numeric conversion completed


,opening_stock,units_received,units_sold,unit_price,discount_pct,days_to_expiry,temperature_c,waste_units,stockout_flag
0,27,0,6,32.0,0.0,1,4.1,0,0
1,30,12,5,32.0,0.0,5,3.7,0,0
2,38,0,14,32.0,0.0,2,4.6,0,0
3,49,0,11,32.0,0.0,0,5.7,0,0
4,19,13,12,32.0,0.0,0,3.0,0,0


In [37]:
invalid_mask = (
    df["product_id"].isna()
    | df["product_id"].eq("")
    | df["category"].isna()
    | df["category"].eq("")
    | df["date"].isna()
    | df[numeric_columns].isna().any(axis=1)
    | (df["opening_stock"] < 0)
    | (df["units_received"] < 0)
    | (df["units_sold"] < 0)
    | (df["unit_price"] < 0)
    | (df["discount_pct"] < 0)
    | (df["discount_pct"] > 100)
    | (df["waste_units"] < 0)
    | (~df["stockout_flag"].isin([0, 1]))
)

invalid_rows = df[invalid_mask].copy()

print("Invalid rows:", len(invalid_rows))
display(invalid_rows.head())

Invalid rows: 0


,product_id,date,category,opening_stock,units_received,units_sold,unit_price,discount_pct,days_to_expiry,temperature_c,waste_units,stockout_flag


In [38]:
df = df.drop_duplicates().copy()

# Recalculate the invalid mask after removing exact duplicates.
invalid_mask = (
    df["product_id"].isna()
    | df["product_id"].eq("")
    | df["category"].isna()
    | df["category"].eq("")
    | df["date"].isna()
    | df[numeric_columns].isna().any(axis=1)
    | (df["opening_stock"] < 0)
    | (df["units_received"] < 0)
    | (df["units_sold"] < 0)
    | (df["unit_price"] < 0)
    | (df["discount_pct"] < 0)
    | (df["discount_pct"] > 100)
    | (df["waste_units"] < 0)
    | (~df["stockout_flag"].isin([0, 1]))
)

invalid_rows = df[invalid_mask].copy()
clean_df = df[~invalid_mask].copy()

# Check for repeated product/date records.
duplicate_keys = clean_df[
    clean_df.duplicated(
        subset=["product_id", "date"],
        keep=False
    )
].copy()

clean_df = clean_df.drop_duplicates(
    subset=["product_id", "date"],
    keep=False
).copy()

clean_df = clean_df.sort_values(
    ["product_id", "date"]
).reset_index(drop=True)

print("Clean rows:", len(clean_df))
print("Invalid rows:", len(invalid_rows))
print("Repeated product/date rows:", len(duplicate_keys))

Clean rows: 1800
Invalid rows: 0
Repeated product/date rows: 0


In [39]:
clean_file = OUT_DIR / "perishable_inventory_clean.csv"

invalid_file = REPORT_DIR / "perishable_inventory_rejected.csv"

duplicate_file = REPORT_DIR / "perishable_inventory_duplicate_dates.csv"

clean_df.to_csv(clean_file, index=False)
invalid_rows.to_csv(invalid_file, index=False)
duplicate_keys.to_csv(duplicate_file, index=False)

report = {
    "original_file": RAW_FILE.name,
    "clean_rows": int(len(clean_df)),
    "invalid_rows": int(len(invalid_rows)),
    "duplicate_product_date_rows": int(len(duplicate_keys)),
    "cleaned_file": str(clean_file)
}

report_file = REPORT_DIR / "perishable_inventory_report.json"

report_file.write_text(
    json.dumps(report, indent=4),
    encoding="utf-8"
)

print("Cleaning completed.")
print("Cleaned dataset:", clean_file)
print("Rejected rows:", invalid_file)
print("Duplicate rows:", duplicate_file)
print(json.dumps(report, indent=4))

Cleaning completed.
Cleaned dataset: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\processed\perishable_inventory_clean.csv
Rejected rows: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\reports\perishable_inventory_rejected.csv
Duplicate rows: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\reports\perishable_inventory_duplicate_dates.csv
{
    "original_file": "perishable_inventory.csv",
    "clean_rows": 1800,
    "invalid_rows": 0,
    "duplicate_product_date_rows": 0,
    "cleaned_file": "D:\\DATA SCIENCE COURSE\\ML\\ML_Projects\\WasteWise-AI\\datasets\\processed\\perishable_inventory_clean.csv"
}
